# 10 · 고속(경량) 실시간 — 신경망 없이 높은 fps

08/09의 실시간이 느린 원인은 **DA(≈570ms) + FastSAM**을 매 프레임 돌리기 때문(≈0.5fps). 이 노트북은 **신경망을 빼고** 순수 ArUco 기하 + 경량 배경차분으로 처리해 **높은 fps**를 낸다.

| 방식 | 검출 | 크기/높이 | 자세 | 속도 |
|---|---|---|---|---|
| 08/09 | DA+FastSAM | ArUco 기하 | DA/기하 | ~0.5 fps |
| **10(이 노트북)** | **경량 배경차분(저해상도)** | ArUco 기하 | 높이로 판별 | **~8~14 fps** |

**속도/정확도 조절:** `SCALE`(처리 해상도 배율)로 조절. 0.4≈14fps, 0.6≈7.6fps, 1.0≈3fps.

> **중요:** 배경차분이라 **`r`로 같은 각도의 빈 보드를 기준으로 잡아야** 깨끗하다. (기준이 부실하면 보드 전체가 한 덩어리로 잡힘.) DA를 안 쓰므로 부피/모양 정밀도는 08/09보다 낮지만 실시간엔 유리.

In [ ]:
%load_ext autoreload
%autoreload 2
import os, sys, glob
import cv2, numpy as np
import matplotlib.pyplot as plt
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(ROOT, 'src'))
import aruco_utils as au, bg_segment as bg, live_fast as lf, scene3d as s3

OUTPUT_DIR = os.path.join(ROOT, 'output'); SCENE_DIR = os.path.join(ROOT, 'data', 'scene_images')
CALIB_DIR = os.path.join(ROOT, 'data', 'calib_images')
SQ = 0.038
board = cv2.aruco.CharucoBoard((5, 7), SQ, SQ*22/30, cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_5X5_1000))
K, dist = au.load_intrinsics(os.path.join(OUTPUT_DIR, 'camera_intrinsics.npz'))

CAM_INDEX = 0
SCALE = 0.5      # 처리 배율(작을수록 빠름/거침). 0.4~0.6 권장
print('ready (신경망 로딩 없음 → 시작도 빠름)')

## 속도 측정 (정지영상)

> 오프라인 기준(각도 다른 캘리브 사진)은 검출이 부정확할 수 있음 — **속도 확인용**. 정확한 검출은 실시간 `r`.

In [ ]:
import time
empty = cv2.imread(sorted(glob.glob(os.path.join(CALIB_DIR, '*.jpg')))[0])
ref = bg.make_reference_canonical(empty, board, K, dist, SQ)
cands = sorted(glob.glob(os.path.join(OUTPUT_DIR, 'snap_raw_*.png'))) or sorted(glob.glob(os.path.join(SCENE_DIR, '*.*')))
img = cv2.imread(cands[-1])
for sc in (1.0, 0.6, 0.4):
    f = lambda: lf.process_frame_fast(img, board, K, dist, ref, scale=sc)
    f(); ts = []
    for _ in range(4):
        s = time.time(); f(); ts.append(time.time()-s)
    ms = np.median(ts)*1000
    print(f'SCALE={sc}: {ms:6.0f} ms  (~{1000/ms:4.1f} fps)')

## 실시간 (고속)

① 보드만 두고 실행 → **`r`** (빈 보드 기준) → ② 물체 올리면 검출.
- **`r`**: 기준 설정/갱신  ·  **`s`**: 스냅  ·  **`q`**: 종료
- 느리면 `SCALE`↓ (예: 0.4), 잔검출이면 `thresh`↑.

In [ ]:
lf.run_live_fast(K, dist, board, square_len=SQ, cam_index=CAM_INDEX, scale=SCALE,
                 snapshot_dir=OUTPUT_DIR)
print('종료됨')

## 튜닝 & 정리

- **fps**: `SCALE`로 조절(0.4≈14fps). 검출이 거칠면 0.6.
- **검출 품질**: 반드시 같은 각도 빈 보드로 `r`. 조명 바뀌면 다시 `r`.
- **정밀도 vs 속도**: 부피/모양이 중요하면 08/09(DA+FastSAM), 실시간 위치·추적이 중요하면 이 노트북.
- 향후: DA/FastSAM을 **백그라운드 스레드로 가끔** 돌려 정밀 갱신 + 이 경량 경로로 매 프레임 표시(하이브리드).